In [5]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('olist.db')

In [6]:
query_category = """
SELECT
    p.product_category_name AS category,
    COUNT(DISTINCT o.order_id) AS total_orders,
    SUM(CASE WHEN o.order_status IN ('canceled','unavailable') THEN 1 ELSE 0 END) AS problem_orders,
    ROUND(100.0 * SUM(CASE WHEN o.order_status IN ('canceled','unavailable') THEN 1 ELSE 0 END)
        / COUNT(DISTINCT o.order_id), 2) AS cancellation_rate_pct
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
GROUP BY p.product_category_name
HAVING COUNT(DISTINCT o.order_id) > 50
ORDER BY cancellation_rate_pct DESC;
"""

df_category = pd.read_sql_query(query_category, conn)
df_category.to_csv('segment_category.csv', index=False)
print(df_category.head(15))



                            category  total_orders  problem_orders  \
0                       dvds_blu_ray            59               2   
1   construcao_ferramentas_seguranca           167               5   
2      construcao_ferramentas_jardim           194               4   
3              instrumentos_musicais           628              11   
4                     telefonia_fixa           217               3   
5             livros_interesse_geral           512               7   
6                 eletrodomesticos_2           234               3   
7                    eletroportateis           630               8   
8                               None          1451              14   
9                     consoles_games          1062              10   
10           fashion_roupa_masculina           112               1   
11                 alimentos_bebidas           227               2   
12                        brinquedos          3886              34   
13             utili

In [7]:
query_tenure = """
WITH seller_first_order AS (
    SELECT
        oi.seller_id,
        MIN(o.order_purchase_timestamp) AS first_order_date
    FROM order_items oi
    JOIN orders o ON oi.order_id = o.order_id
    GROUP BY oi.seller_id
),
order_with_tenure AS (
    SELECT
        o.order_id,
        o.order_status,
        CAST(julianday(o.order_purchase_timestamp) - julianday(sfo.first_order_date) AS INTEGER) AS days_since_seller_start
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN seller_first_order sfo ON oi.seller_id = sfo.seller_id
)
SELECT
    CASE
        WHEN days_since_seller_start < 30 THEN '0-30 days (new)'
        WHEN days_since_seller_start < 90 THEN '30-90 days'
        ELSE '90+ days (established)'
    END AS seller_tenure_bucket,
    COUNT(*) AS total_orders,
    SUM(CASE WHEN order_status IN ('canceled','unavailable') THEN 1 ELSE 0 END) AS problem_orders,
    ROUND(100.0 * SUM(CASE WHEN order_status IN ('canceled','unavailable') THEN 1 ELSE 0 END) / COUNT(*), 2) AS cancellation_rate_pct
FROM order_with_tenure
GROUP BY seller_tenure_bucket
ORDER BY cancellation_rate_pct DESC;
"""

df_tenure = pd.read_sql_query(query_tenure, conn)
df_tenure.to_csv('segment_tenure.csv', index=False)
print(df_tenure)

     seller_tenure_bucket  total_orders  problem_orders  cancellation_rate_pct
0         0-30 days (new)         12253             132                   1.08
1              30-90 days         16451              85                   0.52
2  90+ days (established)         83946             332                   0.40


In [8]:
# Query 4: Delivery Delay
query_delivery = """
SELECT
    CASE
        WHEN julianday(order_delivered_customer_date) - julianday(order_estimated_delivery_date) < 0 THEN 'early'
        WHEN julianday(order_delivered_customer_date) - julianday(order_estimated_delivery_date) = 0 THEN 'on_time'
        ELSE 'late'
    END AS delivery_bucket,
    COUNT(*) AS total_orders,
    SUM(CASE WHEN order_status IN ('canceled','unavailable') THEN 1 ELSE 0 END) AS problem_orders,
    ROUND(100.0 * SUM(CASE WHEN order_status IN ('canceled','unavailable') THEN 1 ELSE 0 END) / COUNT(*), 2) AS cancellation_rate_pct
FROM orders
WHERE order_delivered_customer_date IS NOT NULL
GROUP BY delivery_bucket
ORDER BY cancellation_rate_pct DESC;
"""
df_delivery = pd.read_sql_query(query_delivery, conn)
df_delivery.to_csv('segment_delivery.csv', index=False)
print("=== Delivery Delay ===")
print(df_delivery)

=== Delivery Delay ===
  delivery_bucket  total_orders  problem_orders  cancellation_rate_pct
0            late          7827               1                   0.01
1           early         88649               5                   0.01


In [9]:
# Query 5: Priority Table
query_priority = """
WITH seller_first_order AS (
    SELECT oi.seller_id, MIN(o.order_purchase_timestamp) AS first_order_date
    FROM order_items oi JOIN orders o ON oi.order_id = o.order_id
    GROUP BY oi.seller_id
),
combined AS (
    SELECT
        p.product_category_name AS category,
        CASE
            WHEN julianday(o.order_purchase_timestamp) - julianday(sfo.first_order_date) < 30 THEN 'new_seller'
            ELSE 'established_seller'
        END AS seller_type,
        o.order_id,
        o.order_status
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN products p ON oi.product_id = p.product_id
    JOIN seller_first_order sfo ON oi.seller_id = sfo.seller_id
)
SELECT
    category,
    seller_type,
    COUNT(DISTINCT order_id) AS order_volume,
    ROUND(100.0 * SUM(CASE WHEN order_status IN ('canceled','unavailable') THEN 1 ELSE 0 END) / COUNT(DISTINCT order_id), 2) AS cancel_rate_pct,
    ROUND(COUNT(DISTINCT order_id) * (100.0 * SUM(CASE WHEN order_status IN ('canceled','unavailable') THEN 1 ELSE 0 END) / COUNT(DISTINCT order_id)), 0) AS impact_score
FROM combined
GROUP BY category, seller_type
HAVING order_volume > 30
ORDER BY impact_score DESC
LIMIT 15;
"""
df_priority = pd.read_sql_query(query_priority, conn)
df_priority.to_csv('priority_table.csv', index=False)
print("=== Priority Table ===")
print(df_priority)

=== Priority Table ===
                  category         seller_type  order_volume  cancel_rate_pct  \
0            esporte_lazer  established_seller          6896             0.62   
1   informatica_acessorios  established_seller          5934             0.72   
2    utilidades_domesticas  established_seller          5104             0.84   
3             beleza_saude  established_seller          7731             0.39   
4         moveis_decoracao  established_seller          5949             0.45   
5               brinquedos  established_seller          3500             0.71   
6       relogios_presentes  established_seller          5281             0.36   
7                    bebes  established_seller          2548             0.71   
8               automotivo  established_seller          3348             0.51   
9          cama_mesa_banho  established_seller          8788             0.18   
10              automotivo          new_seller           557             2.51   
11   